In [ ]:
import logging
import os
from collections import defaultdict
import xarray as xr
import numpy as np
import pcraster as pcr
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import geopandas as gpd
from scores.continuous import kge
from scores.continuous import nse
import resevoir_functions_Daan as rf
import PointModelDaan as pm
from sklearn.metrics import root_mean_squared_error
import importlib
importlib.reload(pm)

import cartopy.crs as ccrs, cartopy.feature as cfeature
from matplotlib.colors import LogNorm
import matplotlib.lines as mlines


#set up logging and correct file path
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
logger = logging.getLogger(__name__)
logging.disable(logging.INFO)
plt.rcParams.update({'font.size': 11, 'axes.titlesize': 11, 'savefig.dpi': 300, 'savefig.bbox': 'tight'})


In [ ]:
from pathlib import Path

DATA_ROOT = Path('Data/ResOpsUS/ResOpsUS')
TS_DIR    = DATA_ROOT / 'raw_time_series'

# Build lookup table: dam_id → agency → filename
file_records = []
for _f in sorted(TS_DIR.glob('*.csv')):
    _parts = _f.stem.rsplit('_', 1)
    file_records.append({'dam_id': int(_parts[0]), 'agency': _parts[1], 'filename': _f.name})
file_df = pd.DataFrame(file_records)

print(f'ResOpsUS files indexed: {len(file_df)}')

# Agency preference order (fallback if completeness is tied)
AGENCY_PREF = ['USBR', 'BOR', 'USBRPN', 'USACE', 'CDEC', 'USGS', 'TWDB', 'JCS']

def load_resopsus(grand_id, variables=('storage', 'outflow'), start='1979', end='2021'):
    """Pick the agency file with the MOST non-NaN storage+outflow within [start, end]."""
    candidates = file_df[file_df['dam_id'] == grand_id].copy()
    if candidates.empty:
        raise FileNotFoundError(f'No ResOpsUS file found for GRanD ID {grand_id}.')

    if len(candidates) == 1:
        chosen = candidates.iloc[0]
    else:
        scores = []
        for _, row in candidates.iterrows():
            d = pd.read_csv(TS_DIR / row['filename'], parse_dates=['date'], index_col='date').loc[start:end]
            coverage = sum(d[v].notna().sum() for v in variables if v in d.columns)
            pref = AGENCY_PREF.index(row['agency']) if row['agency'] in AGENCY_PREF else 99
            scores.append((-coverage, pref, row['filename']))     # -coverage → most-covered first
        scores.sort()
        chosen = candidates[candidates['filename'] == scores[0][2]].iloc[0]
        print(f'Multiple files: {", ".join(candidates["agency"])} → using {chosen["agency"]} (most in-window coverage)')

    df = pd.read_csv(TS_DIR / chosen['filename'], parse_dates=['date'], index_col='date')
    return df, chosen['agency']

def resample_monthly(df, agg='mean'):
    """
    Resample a daily ResOpsUS DataFrame to monthly.
    Storage/elevation → end-of-month value; flows/evap → monthly mean then convert.
    Returns storage in MCM/month, flows in m³/month (×seconds in month).
    """
    monthly = pd.DataFrame(index=df.resample('MS').mean().index)

    # Storage and elevation: end-of-month snapshot
    for col in ('storage', 'elevation'):
        if col in df.columns:
            monthly[col] = df[col].resample('MS').last()

    # Flows: mean daily m³/s → total m³/month
    for col in ('inflow', 'outflow'):
        if col in df.columns:
            days_in_month = monthly.index.days_in_month
            monthly[col + '_m3s']      = df[col].resample('MS').mean()
            monthly[col + '_m3month']  = monthly[col + '_m3s'] * 86400 * days_in_month

    return monthly

In [ ]:
dam_id = 17393
bridge  = pd.read_excel('Data/GeoDAR_PCRGLOBWB.xlsx')
dam_row = bridge[bridge['geodar_v11'] == dam_id].iloc[0]
GRAND_ID = dam_row['grand_id'] 
latitude   =  dam_row['geodar_lat']     #pour-point latitude  [°N]
longitude  =  dam_row['geodar_long']    #pour-point longitude [°E]
capacity_design = dam_row['capacity'] * 1e6 #design capacity: million m³ → m³
dam_row

In [ ]:

# Hoover Dam / Lake Mead
df_daily, agency_used = load_resopsus(GRAND_ID)
print(f'Agency used : {agency_used}')
print(f'Date range  : {df_daily.index[0].date()} → {df_daily.index[-1].date()}')
print(f'Shape       : {df_daily.shape}')
print()
print('Missing values per column:')
print(df_daily.isna().sum().to_string())
print()
df_daily.describe()



In [ ]:
def build_compare(dam_id, start='1979', end='2021'):
    bridge = pd.read_excel('Data/GeoDAR_PCRGLOBWB.xlsx')
    row = bridge[bridge['geodar_v11'] == dam_id].iloc[0]
    grand_id = row['grand_id']

    df, capacity, use_type = pm.get_data(dam_id, plot=False)
    sim = pm.run_pointmodel(dam_id, df, capacity, use_type, plot=False)
    s = sim.copy(); s['time'] = pd.to_datetime(s['time']); s = s.set_index('time')
    s.index = s.index.to_period('M').to_timestamp()

    dd, _ = load_resopsus(grand_id)
    obs = resample_monthly(dd)

    c = pd.DataFrame({
        'obs_outflow': obs['outflow_m3month'],
        'model_outflow': s['model_release'],        # fix 4: total release incl. spill
        'pcr_outflow': s['discharge_m3'],
        'obs_storage': obs['storage'] * 1e6,       # MCM -> m3
        'model_storage': s['modelled_storage'],      # fix 5: end-of-month storage
        'pcr_storage': s['storage_m3'],
    }).loc[start:end].dropna(subset=['obs_outflow', 'obs_storage'])

    for src in ('obs', 'model', 'pcr'):
        c[f'{src}_storage_frac'] = c[f'{src}_storage'] / capacity
        c[f'{src}_outflow_frac'] = c[f'{src}_outflow'] / capacity
    
    c['season'] = c.index.month.map({
    12: 'DJF', 1: 'DJF', 2: 'DJF',
    3: 'MAM', 4: 'MAM', 5: 'MAM',
    6: 'JJA', 7: 'JJA', 8: 'JJA',
    9: 'SON', 10: 'SON', 11: 'SON'
})

    return c, capacity

# df_test, capacity = build_compare(17393)
# display(df_test)

In [ ]:
def pbias(observed, modeled):
    return 100 * (modeled - observed).sum() / observed.sum()

def validation_metrics(df, dam_id=None, do_pcr=False, do_seasonal_components=False, min_months=12):
    season_map = {12:'DJF',1:'DJF',2:'DJF', 3:'MAM',4:'MAM',5:'MAM',
                  6:'JJA',7:'JJA',8:'JJA', 9:'SON',10:'SON',11:'SON'}
    seasons = ['DJF', 'MAM', 'JJA', 'SON']
    sources = ['model'] + (['pcr'] if do_pcr else [])   # label == column prefix

    result = {}
    for var in ['storage', 'outflow']:
        for src in sources:
            # clean per obs/model pair: drop inf and NaN together
            pair = (df[[f'obs_{var}_frac', f'{src}_{var}_frac']]
                      .replace([np.inf, -np.inf], np.nan)
                      .dropna())
            if len(pair) < min_months:
                continue                                  # not enough obs -> skip this variable

            obs = xr.DataArray(pair[f'obs_{var}_frac'].values, dims='time')
            mod = xr.DataArray(pair[f'{src}_{var}_frac'].values, dims='time')

            k = kge(mod, obs, include_components=True)
            result[f'{src}_{var}_kge']   = float(k['kge'])
            result[f'{src}_{var}_rho']   = float(k['rho'])
            result[f'{src}_{var}_beta']  = float(k['beta'])
            result[f'{src}_{var}_alpha'] = float(k['alpha'])
            result[f'{src}_{var}_nse']   = float(nse(mod, obs))
            result[f'{src}_{var}_pbias'] = float(pbias(obs, mod))
            result[f'{src}_{var}_rmse']  = float(root_mean_squared_error(obs, mod))
            result[f'{src}_{var}_n']     = len(pair)

            season_arr = pair.index.month.map(season_map)
            for season in seasons:
                ps = pair[season_arr == season]
                if len(ps) < 3:
                    continue
                obs_s = xr.DataArray(ps[f'obs_{var}_frac'].values, dims='time')
                mod_s = xr.DataArray(ps[f'{src}_{var}_frac'].values, dims='time')
                ks = kge(mod_s, obs_s, include_components=True)
                result[f'{src}_{var}_kge_{season}'] = float(ks['kge'])
                if do_seasonal_components:
                    result[f'{src}_{var}_alpha_{season}'] = float(ks['alpha'])
                    result[f'{src}_{var}_beta_{season}']  = float(ks['beta'])
                    result[f'{src}_{var}_rho_{season}']   = float(ks['rho'])

    return pd.Series(result, name=dam_id)


# metric = validation_metrics(df_test, do_pcr=True)
# display(metric)



In [ ]:
bridge = pd.read_excel('Data/GeoDAR_PCRGLOBWB.xlsx')
resops_ids = set(file_df['dam_id'])                       # grand_ids with a ResOpsUS file
candidates = bridge[bridge['grand_id'].isin(resops_ids)]
dam_ids = candidates['geodar_v11'].tolist()
print(len(dam_ids), 'dams have both PCR point data and a ResOpsUS record')

In [ ]:
def validate_dams(dam_ids, do_pcr=True, do_seasonal_components=False):
    rows, failures = [], {}
    for d in dam_ids:
        try:
            c, _ = build_compare(d)
            if len(c) < 12:                               # too little overlap to score
                raise ValueError(f'only {len(c)} overlapping months')
            rows.append(validation_metrics(c, dam_id=d, do_pcr=do_pcr,
                                           do_seasonal_components=do_seasonal_components))
        except Exception as e:
            failures[d] = str(e)
    metrics_df = pd.DataFrame(rows)
    if not metrics_df.empty:
        metrics_df.index.name = 'dam_id'
    print(f'{len(rows)} ok, {len(failures)} failed')
    for d, e in failures.items():
        print(f'  dam {d}: {e}')
    return metrics_df, failures

# test on a handful first, then the full list
# metrics_df, failures = validate_dams(dam_ids[:5])
# metrics_df.to_csv('validation_metrics.csv')

In [ ]:
inv = pd.read_csv('dam_inventory.csv')
complete = inv[(inv.storage_days > 3650) & (inv.outflow_days > 3650)].copy()   # >~10 yr of each
complete['minday'] = complete[['storage_days', 'outflow_days']].min(axis=1)

N = 5
groups = ['Hydroelectricity', 'Irrigation', 'Water Supply', 'Flood Control']
shortlist = (complete[complete['use'].isin(groups)]
             .sort_values('minday', ascending=False)
             .groupby('use', group_keys=False)
             .head(N))

# force-include named dams (both pass the completeness bar; 16264 excluded — no outflow record)
must = [17393, 16247]                                   # Hoover, Glen Canyon
shortlist = pd.concat([shortlist, inv[inv.geodar.isin(must)]]).drop_duplicates('geodar')

dam_ids   = shortlist['geodar'].tolist()
use_map   = dict(zip(shortlist['geodar'], shortlist['use']))
cap_map   = dict(zip(shortlist['geodar'], shortlist['cap_MCM']))
print(shortlist[['geodar','use','cap_MCM','storage_days','outflow_days']]
      .sort_values(['use','cap_MCM']).to_string(index=False))

In [ ]:
metrics_df, failures = validate_dams(dam_ids, do_pcr=True)
metrics_df['use']     = metrics_df.index.map(use_map)
metrics_df['cap_MCM'] = metrics_df.index.map(cap_map)
metrics_df.to_csv('validation_metrics.csv')
print(f'{len(metrics_df)} dams scored, {len(failures)} failed')

In [ ]:
metricsdf = pd.read_csv('usable_dams.csv'); metricsdf['region'] = 'west'
metricseast = pd.read_csv('usable_dams_east.csv')

descriptive_west = pd.merge(metricsdf, bridge, left_on ='geodar', right_on='geodar_v11' )
descriptive_east = pd.merge(metricseast, bridge, left_on ='geodar', right_on='geodar_v11')


descriptivedf = pd.concat([descriptive_west, descriptive_east], ignore_index=True)

q1, q2 = descriptivedf['cap_MCM'].quantile([1/3, 2/3])
descriptivedf['size_bin'] = pd.cut(descriptivedf['cap_MCM'],
                                   [-1, q1, q2, 1e12],
                                   labels=['small', 'medium', 'large'])


print(descriptivedf['geodar_lat'].min(), descriptivedf['geodar_lat'].max())
print(descriptivedf['geodar_long'].min(), descriptivedf['geodar_long'].max())
descriptivedf.isna().sum()


In [ ]:
descriptivedf[descriptivedf['geodar'] == 20636]

In [ ]:
file_path = os.path.join(os.getcwd(), 'images') + os.sep


mk = {'Irrigation':'o','Flood Control':'s','Hydroelectricity':'^',
      'Water Supply':'D','Other':'X'}
norm = LogNorm(vmin=descriptivedf.cap_MCM.min(), vmax=descriptivedf.cap_MCM.max())  # gedeelde schaal

fig = plt.figure(figsize=(9,6))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_extent([-125,-78,25,50])

ax.add_feature(cfeature.OCEAN.with_scale('50m'), facecolor='#dbeaf2')
ax.add_feature(cfeature.LAND.with_scale('50m'),  facecolor='#f6f4ee')
ax.add_feature(cfeature.LAKES.with_scale('10m'), facecolor='#dbeaf2', edgecolor='none')
ax.add_feature(cfeature.RIVERS.with_scale('10m'), edgecolor='#9ec7e0', linewidth=.6)
ax.add_feature(cfeature.STATES.with_scale('50m'), edgecolor='gray', linewidth=.5)
ax.add_feature(cfeature.COASTLINE.with_scale('50m'), linewidth=.6)

# markervorm = use type, kleur = capaciteit (log)
for use, m in mk.items():
    g = descriptivedf[descriptivedf['use_x'] == use]
    if len(g) == 0: continue
    sc = ax.scatter(g.geodar_long, g.geodar_lat, c=g.cap_MCM, norm=norm, cmap='inferno',
                    marker=m, s=20, edgecolor='k', linewidth=.3, zorder=5,
                    transform=ccrs.PlateCarree())
plt.colorbar(sc, label='capacity [MCM]', shrink=.7)

# vorm-legenda (zwart, want kleur codeert capaciteit)
handles = [mlines.Line2D([],[],color='k',marker=m,linestyle='None',markersize=7,label=u)
           for u,m in mk.items() if u in descriptivedf['use_x'].unique()]
ax.legend(handles=handles, title='use type', loc='lower left', fontsize=8, framealpha=.9)

# gridlines
gl = ax.gridlines(draw_labels=True, linewidth=.3, color='gray', alpha=.4)
gl.top_labels = False; gl.right_labels = False

# noordpijl (PlateCarree is noord-omhoog, dus een verticale pijl klopt)
ax.annotate('N', xy=(0.96, 0.93), xytext=(0.96, 0.83),
            arrowprops=dict(facecolor='black', width=3.5, headwidth=11),
            ha='center', va='center', fontsize=12, fontweight='bold',
            xycoords='axes fraction', zorder=6)

# CRS-vermelding
ax.text(0.99, 0.01, 'CRS: WGS84 (EPSG:4326), PlateCarree',
        transform=ax.transAxes, ha='right', va='bottom', fontsize=7, color='gray')

plt.title('357 validation reservoirs by use type (western/central CONUS)')
plt.tight_layout()
plt.savefig(file_path + 'validation_dam_map.pdf')
plt.show()

In [ ]:
file_path = os.path.join(os.getcwd(), 'images') + os.sep
from matplotlib.colors import LogNorm

#fig 1
sns.histplot(data=descriptivedf, x='use_x', hue='use_x', palette= 'inferno', shrink=0.7)
plt.tight_layout()
plt.title('357 validation dams by use type')
plt.xlabel('use type')
plt.tight_layout()
plt.savefig(file_path + 'usetypehist.pdf')

plt.show()

#fig 2
sns.histplot(data=descriptivedf, x='size_bin', hue='use_x', multiple='dodge', shrink=0.7, palette='inferno')
plt.tight_layout()
plt.title('357 validation dams by size and use type')
plt.xlabel('size')
plt.tight_layout()
plt.savefig(file_path + 'sizebinhist.pdf')
plt.show()

#fig 3
descriptivedf['storage_yr'] = descriptivedf['storage_days'] / 365
descriptivedf['outflow_yr'] = descriptivedf['outflow_days'] / 365

fig, ax = plt.subplots(1, 2, figsize=(11, 4), sharey=True)

sns.histplot(data=descriptivedf, x='storage_yr', hue='use_x', multiple='stack',
             palette='inferno', bins=20, ax=ax[0], legend=False)
ax[0].set_xlabel('storage record [yr]')

sns.histplot(data=descriptivedf, x='outflow_yr', hue='use_x', multiple='stack',
             palette='inferno', bins=20, ax=ax[1])
ax[1].set_xlabel('outflow record [yr]')

fig.suptitle('Storage and outflow record length by use type')
fig.tight_layout()
plt.savefig(file_path + 'recordlength.pdf')
plt.show()
file_path = os.path.join(os.getcwd(), 'Data', 'POINTDATA') + os.sep


In [ ]:
from pathlib import Path
import pandas as pd, numpy as np, matplotlib.pyplot as plt
TS = Path('Data/ResOpsUS/ResOpsUS/raw_time_series')
u  = pd.concat([pd.read_csv('usable_dams.csv').assign(region='west'), pd.read_csv('usable_dams_east.csv')], ignore_index=True)
fdf = pd.DataFrame([{'dam_id':int(f.stem.rsplit('_',1)[0]),'filename':f.name}
                    for f in sorted(TS.glob('*.csv'))])
YEARS = list(range(1979, 2022))

def dam_year_sets(gid):                         # jaren met data, uit het best-gedekte bestand
    cand = fdf[fdf.dam_id == gid]; bc=-1; bs=set(); bo=set()
    for fn in cand.filename:
        d = pd.read_csv(TS/fn, usecols=lambda c: c in ('date','storage','outflow'), dtype=str)
        yr = d['date'].str.slice(0,4).astype(int); d, yr = d[(yr>=1979)&(yr<=2021)], yr[(yr>=1979)&(yr<=2021)]
        sto = d['storage'].replace('NA', np.nan) if 'storage' in d else pd.Series(dtype=object)
        out = d['outflow'].replace('NA', np.nan) if 'outflow' in d else pd.Series(dtype=object)
        if sto.notna().sum()+out.notna().sum() > bc:
            bc = sto.notna().sum()+out.notna().sum()
            bs, bo = set(yr[sto.notna()]), set(yr[out.notna()])
    return bs, bo

sc = {y:0 for y in YEARS}; oc = {y:0 for y in YEARS}
for gid in u['grand_id']:
    s, o = dam_year_sets(gid)
    for y in s: sc[y] = sc.get(y,0)+1
    for y in o: oc[y] = oc.get(y,0)+1
cov = pd.DataFrame({'year':YEARS, 'storage':[sc[y] for y in YEARS], 'outflow':[oc[y] for y in YEARS]})

fig, ax = plt.subplots(figsize=(9,4))
ax.fill_between(cov.year, cov.storage, alpha=.35, color='#1f77b4'); ax.plot(cov.year, cov.storage, color='#1f77b4', label='storage')
ax.fill_between(cov.year, cov.outflow, alpha=.35, color='#d62728'); ax.plot(cov.year, cov.outflow, color='#d62728', label='outflow')
ax.set(xlabel='year', ylabel='reservoirs with data', ylim=(0,357),
       title='Temporal data coverage of the 357 validation reservoirs'); ax.legend()
plt.tight_layout()
plt.savefig(file_path + 'temporal_data_coverage.pdf')
plt.show()
#de validatie is dun in de vroege jaren '80 en het rijkst ~1995–2019 — feitelijk vooral een 1990–2020-validatie. Die afval-rand rechts wil je benoemen, anders denkt een lezer dat er recent iets misgaat.

In [ ]:
u = pd.concat([pd.read_csv('usable_dams.csv').assign(region='west'),
               pd.read_csv('usable_dams_east.csv')], ignore_index=True)
u['use'] = u['use'].fillna('Unclassified')      # the 12 dams with no primary use

def summ(g):
    return pd.Series({
        'n': len(g),
        'cap_median_MCM': round(g.cap_MCM.median(), 1),
        'cap_min_MCM':    round(g.cap_MCM.min(), 1),
        'cap_max_MCM':    round(g.cap_MCM.max(), 1),
        'storage_yr_median': round((g.storage_days/365).median(), 1),
        'outflow_yr_median': round((g.outflow_days/365).median(), 1),
    })

order = ['Flood Control', 'Irrigation', 'Water Supply', 'Hydroelectricity',
         'Recreation', 'Navigation', 'Other', 'Unclassified']

tab = u.groupby('use').apply(summ, include_groups=False).reindex(order)
tab = pd.concat([tab, summ(u).to_frame('All').T])
tab['n'] = tab['n'].astype(int)
tab.to_csv(file_path + 'Table1_sample.csv')
tab.to_latex(file_path + 'Table1.tex')
tab
#Hydroelectricity en Water Supply zijn groter (mediaan ~300–340 MCM, met de twee reuzen Glen Canyon/Hoover als max) én hebben langere reeksen (~40 jr vs ~28–34 jr). Die regel capaciteit-mediaan per type laat de size/use-verstrengeling in harde getallen zien — precies de nuance die je in tekst wilt benoemen.

In [ ]:
usable = pd.concat([pd.read_csv('usable_dams.csv').assign(region='west'), pd.read_csv('usable_dams_east.csv')], ignore_index=True)
dam_ids = usable['geodar'].tolist()

metrics_df, failures = validate_dams(dam_ids, do_pcr=True)

tags = usable.set_index('geodar')[['use', 'cap_MCM', 'size_bin']]   # stratification labels
metrics_df = metrics_df.join(tags)
metrics_df.to_csv('validation_metrics_all.csv')
print(f'{len(metrics_df)} scored, {len(failures)} failed')   # expect 357 scored, 0 failed

In [ ]:
metrics_df= pd.read_csv('validation_metrics_all.csv')

irrigation_types = {'Irrigation', 'Water Supply'}

metrics_df['use'].unique()

metrics_df['path'] = metrics_df['path'] = np.where(metrics_df['use'].isin(irrigation_types), 'Irrigation', 'Hydroelectricity')
metrics_df[metrics_df['model_storage_kge'] == metrics_df['model_storage_kge'].min()]
# after concatenating west + east into metrics_df:
metrics_df['metric_ok'] = ~((metrics_df['model_storage_beta'].abs() > 3) |
                            (metrics_df['model_storage_pbias'].abs() > 200) |
                            (metrics_df['model_storage_kge'] < -5) |
                            (metrics_df['model_storage_beta'] < 0))
print('usable:', metrics_df['metric_ok'].sum(), '| excluded:', (~metrics_df['metric_ok']).sum())
metrics_df['size_bin'] = pd.cut(metrics_df['cap_MCM'],
                                   [-1, q1, q2, 1e12],
                                   labels=['small', 'medium', 'large'])
metrics_df


In [ ]:
metrics_df.to_csv('357metrics.csv')

In [ ]:
metrics_df.loc[~metrics_df.metric_ok, 'use'].value_counts()

In [ ]:
c, cap = build_compare(21629)          # df_compare met de *_storage_frac kolommen

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(c.index, c['obs_storage_frac'],   color='k',       lw=1.6, label='observed')
ax.plot(c.index, c['model_storage_frac'], color='#d62728', lw=1.2, label='point model (STARFIT)')
ax.plot(c.index, c['pcr_storage_frac'],   color='#1f77b4', lw=1.0, alpha=.7, label='PCR-GLOBWB2')
ax.axhline(0, color='gray', lw=.5)
ax.set_ylabel('storage / capacity'); ax.set_xlabel('year')
ax.set_title(f'Flood-control reservoir kept near-empty in reality, filled by the model '
             f'(dam 21629, {cap/1e6:.0f} MCM)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(os.getcwd(), 'images', 'floodcontrol_example.pdf'))
plt.show()
metrics_df[metrics_df['model_storage_kge'] == metrics_df['model_storage_kge'].min()]


In [ ]:
rows = []
for var in ['storage', 'outflow']:
    for src in ['model', 'pcr']:
        sub = metrics_df[metrics_df['metric_ok']].dropna(subset=[f'{src}_{var}_kge'])
        rows.append({
            'variable': var, 'source': src, 'n': len(sub),
            'KGE':  round(sub[f'{src}_{var}_kge'].median(), 2),
            'r':    round(sub[f'{src}_{var}_rho'].median(), 2),
            'beta': round(sub[f'{src}_{var}_beta'].median(), 2),
            'alpha':round(sub[f'{src}_{var}_alpha'].median(), 2),
            'NSE':  round(sub[f'{src}_{var}_nse'].median(), 2),
            'PBIAS':round(sub[f'{src}_{var}_pbias'].median(), 1),
            'RMSE': round(sub[f'{src}_{var}_rmse'].median(), 3),
        })
summary = pd.DataFrame(rows)
summary.to_csv('results_summary_table.csv', index=False)
summary.to_latex('results_summary_table.tex', index=False)   # for the thesis
summary

In [ ]:

file_path = os.path.join(os.getcwd(), 'images') + os.sep 

clean   = metrics_df[metrics_df['metric_ok']].copy()       # 299 dams for distributions/aggregates
KGE_REF = 1 - 2**0.5                                        # mean-flow benchmark ≈ -0.41
path_order = list(clean['path'].dropna().unique())
pal_path   = dict(zip(path_order, sns.color_palette('Set2', len(path_order))))

# long format for the ECDF / component / seasonal figures
id_vars = ['dam_id', 'use', 'cap_MCM', 'size_bin', 'metric_ok', 'path']
long = metrics_df.melt(id_vars=id_vars, var_name='col', value_name='value')
SEASONS = {'DJF', 'MAM', 'JJA', 'SON'}
def _parse(c):
    p = c.split('_'); src, var, rest = p[0], p[1], p[2:]
    if rest and rest[-1] in SEASONS: season, metric = rest[-1], '_'.join(rest[:-1])
    else:                            season, metric = 'annual', '_'.join(rest)
    return pd.Series([src, var, metric, season])
long[['source', 'variable', 'metric', 'season']] = long['col'].apply(_parse)
long = long.drop(columns='col')
long_ok = long[long['metric_ok']]

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
for a, var in zip(ax, ['storage', 'outflow']):
    sns.scatterplot(data=clean, x=f'pcr_{var}_kge', y=f'model_{var}_kge',
                    hue='path', palette=pal_path, s=45, edgecolor='k', linewidth=.3, ax=a)
    a.axline((0, 0), (1, 1), ls='--', c='k', lw=.8)
    a.set(xlim=(-2, 1), ylim=(-2, 1), xlabel=f'PCR-GLOBWB {var} KGE', ylabel=f'point model {var} KGE')
    a.set_title(f'{var}: on 1:1 ⇒ point model reproduces PCR'); a.legend(fontsize=7)
plt.tight_layout(); plt.savefig(file_path + 'res1_model_vs_pcr.pdf'); plt.show()

d = long_ok[(long_ok.metric == 'kge') & (long_ok.season == 'annual')]
fig, ax = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
for a, var in zip(ax, ['storage', 'outflow']):
    sns.ecdfplot(data=d[d.variable == var], x='value', hue='source', ax=a)
    a.axvline(0, ls=':', c='gray'); a.axvline(KGE_REF, ls='--', c='r', lw=.8)
    a.set(xlim=(-2, 1), xlabel=f'{var} KGE'); a.set_title(f'{var} KGE — model vs PCR')
plt.tight_layout(); plt.savefig(file_path + 'res2_kge_ecdf.pdf'); plt.show()

In [ ]:
comps = ['rho', 'beta', 'alpha']
d = long_ok[(long_ok.source == 'model') & (long_ok.metric.isin(comps)) & (long_ok.season == 'annual')]
fig, ax = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
for a, var in zip(ax, ['storage', 'outflow']):
    dv = d[d.variable == var]
    sns.boxplot(data=dv, x='metric', y='value', order=comps, ax=a, color='lightgray', fliersize=0)
    sns.stripplot(data=dv, x='metric', y='value', order=comps, ax=a, color='k', size=3, alpha=.4, jitter=.2)
    a.axhline(1, ls='--', c='r', lw=.8)
    a.set(ylim=(-0.5, 2.5), xlabel=''); a.set_xticklabels(['r (timing)', 'β (bias)', 'α (variability)'])
    a.set_title(f'{var} KGE components')
ax[0].set_ylabel('component value (1 = perfect)')
plt.tight_layout(); plt.savefig(file_path + 'res3_kge_components.pdf'); plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
for a, var in zip(ax, ['storage', 'outflow']):
    sns.scatterplot(data=clean, x='cap_MCM', y=f'model_{var}_beta',
                    hue='path', palette=pal_path, s=45, edgecolor='k', linewidth=.3, ax=a)
    a.set_xscale('log'); a.axhline(1, ls=':', c='k'); a.set_ylim(0, 2.2)
    bins = np.logspace(np.log10(clean.cap_MCM.min()), np.log10(clean.cap_MCM.max()), 7)
    med = clean.groupby(pd.cut(clean.cap_MCM, bins), observed=True)[f'model_{var}_beta'].median()
    a.plot([iv.mid for iv in med.index], med.values, 'k-o', lw=2, ms=5, label='median')
    a.set(xlabel='capacity [MCM, log]', ylabel='β'); a.set_title(f'{var} bias vs size'); a.legend(fontsize=7)
plt.tight_layout(); plt.savefig(file_path + 'res4_beta_vs_capacity.pdf'); plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 5), sharey=True)
for a, var in zip(ax, ['storage', 'outflow']):
    sns.boxplot(data=clean, x='path', y=f'model_{var}_kge', order=path_order,
                hue='path', palette=pal_path, legend=False, ax=a, fliersize=0)
    sns.stripplot(data=clean, x='path', y=f'model_{var}_kge', order=path_order,
                  ax=a, color='k', size=3, alpha=.4)
    a.axhline(0, ls=':', c='gray'); a.axhline(KGE_REF, ls='--', c='r', lw=.8)
    a.set(ylim=(-2, 1), xlabel=''); a.set_title(f'{var} KGE by operating path')
ax[0].set_ylabel('KGE')
plt.tight_layout(); plt.savefig(file_path + 'res5_kge_by_path.pdf'); plt.show()

In [ ]:
# make sure region is on the frame
if 'region' not in clean.columns:
    west_ids = set(pd.read_csv('usable_dams.csv')['geodar'])
    clean = clean.copy()
    clean['region'] = np.where(clean['dam_id'].isin(west_ids), 'west', 'Mississippi')

regions = ['west', 'Mississippi']
fig, ax = plt.subplots(2, 2, figsize=(11, 9), sharey=True)
for i, reg in enumerate(regions):
    g = clean[clean['region'] == reg]
    for j, var in enumerate(['storage', 'outflow']):
        a = ax[i, j]
        sns.boxplot(data=g, x='path', y=f'model_{var}_kge', order=path_order,
                    hue='path', palette=pal_path, legend=False, ax=a, fliersize=0)
        sns.stripplot(data=g, x='path', y=f'model_{var}_kge', order=path_order,
                      ax=a, color='k', size=3, alpha=.4)
        a.axhline(0, ls=':', c='gray'); a.axhline(KGE_REF, ls='--', c='r', lw=.8)
        a.set(ylim=(-2, 1), xlabel=''); a.set_title(f'{reg} — {var} KGE by path')
ax[0, 0].set_ylabel('KGE'); ax[1, 0].set_ylabel('KGE')
plt.tight_layout(); plt.savefig(file_path + 'res5_kge_by_path_region.pdf'); plt.show()

In [ ]:
d = long_ok[(long_ok.source == 'model') & (long_ok.metric == 'kge') & (long_ok.season.isin(SEASONS))]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for a, var in zip(ax, ['storage', 'outflow']):
    piv = (d[d.variable == var].pivot_table(index='path', columns='season', values='value', aggfunc='median')
                                .reindex(columns=['DJF', 'MAM', 'JJA', 'SON']))
    sns.heatmap(piv, annot=True, fmt='.2f', cmap='RdYlBu', center=0, vmin=-1, vmax=1, ax=a)
    a.set(xlabel='season', ylabel=''); a.set_title(f'{var} seasonal KGE (median)')
plt.tight_layout(); plt.savefig(file_path + 'res6_seasonal_kge.pdf'); plt.show()

In [ ]:
file_path = os.path.join(os.getcwd(), 'images') + os.sep

def monthly_clim(series):
    """Long-run mean by calendar month, broadcast back onto the monthly index."""
    clim = series.groupby(series.index.month).mean()
    return pd.Series(series.index.month.map(clim), index=series.index)

dams = {20889: 'good  (storage KGE = 0.59)',
        20548: 'median (0.00)',
        20074: 'poor  (-1.89)'}

fig, ax = plt.subplots(len(dams), 2, figsize=(14, 3.2 * len(dams)))
for i, (d, lab) in enumerate(dams.items()):
    c, _ = build_compare(d)
    for j, var in enumerate(['storage', 'outflow']):
        a = ax[i, j]
        a.plot(c.index, c[f'obs_{var}_frac'],   color='k',       lw=1.4, label='observed')
        a.plot(c.index, c[f'model_{var}_frac'],  color='#d62728', lw=1.0, label='point model')
        a.plot(c.index, c[f'pcr_{var}_frac'],    color='#1f77b4', lw=0.9, alpha=.6, label='PCR-GLOBWB')
        a.plot(c.index, monthly_clim(c[f'model_{var}_frac']),
               color='#d62728', lw=0.8, ls='--', alpha=.8, label='model climatology')
        if i == 0: a.set_title(var)
        if j == 0: a.set_ylabel('fraction of capacity')
    ax[i, 0].text(0.01, 0.93, f'dam {d} — {lab}', transform=ax[i, 0].transAxes,
                  fontsize=8, fontweight='bold', va='top')
ax[0, 1].legend(fontsize=7, loc='upper right')
fig.suptitle('Observed vs point model vs PCR-GLOBWB')
a.set_xlim(pd.Timestamp('1979'), pd.Timestamp('2022'))
fig.tight_layout()
plt.savefig(file_path + 'res7_hydrographs.pdf'); plt.show()

In [ ]:
#compute inflow-to-band ratio (lean)
import xarray as xr, numpy as np
metrics_df['lat'] = metrics_df['dam_id'].map(bridge.set_index('geodar_v11')['geodar_lat'])
metrics_df['lon'] = metrics_df['dam_id'].map(bridge.set_index('geodar_v11')['geodar_long'])
la = xr.DataArray(metrics_df['lat'].values, dims='d')
lo = xr.DataArray(metrics_df['lon'].values, dims='d')

# mean monthly inflow (m3). If it's slow/heavy, add .isel(time=slice(0,None,10)) before .mean, both for eastern and western CONUS.
def _read_inflow(path):
    return (xr.open_dataset(path)['lake_and_reservoir_inflow']
              .sel(lat=la, lon=lo, method='nearest')
              .isel(time=slice(0, None, 10)).mean('time').values * 86400 * 30.44)

inf_w = _read_inflow('Data/POINTDATA/lake_and_reservoir_inflow_dailyTot_output.nc')
inf_e = _read_inflow('Data/M44/netcdf/lake_and_reservoir_inflow_dailyTot_output.nc')
inf = np.where(np.isfinite(inf_w), inf_w, inf_e)     # western run first, fall back to M44

# active-storage band width (fraction of capacity) from 12 mid-month RF files
fsum = csum = 0
for mo in range(1, 13):
    db = xr.open_dataset(f'Data/POINTDATA/10_param_RF_bounds_final/2000-{mo}-15.nc')
    fsum += db['flood'].sel(latitude=la, longitude=lo, method='nearest').values.squeeze()
    csum += db['conservation'].sel(latitude=la, longitude=lo, method='nearest').values.squeeze()

cap = metrics_df['cap_MCM'].values * 1e6
metrics_df['inflow_to_band'] = (inf / cap) / ((fsum - csum) / 12 / 100)
metrics_df.replace([np.inf, -np.inf], np.nan, inplace=True)
metrics_df.to_csv('metricswithinflowratio.csv')

In [ ]:
# ===== FIG res8 =====
metrics_df = pd.read_csv('metricswithinflowratio.csv')
clean8 = metrics_df[metrics_df['metric_ok']].dropna(subset=['inflow_to_band'])
fig, ax = plt.subplots(1, 2, figsize=(13, 5))

sns.scatterplot(data=clean8, x='inflow_to_band', y='model_storage_alpha',
                hue='path', palette=pal_path, ax=ax[0])
ax[0].set(xscale='log', xlabel='inflow-to-band ratio (log)', ylabel='storage α', title='variability vs ratio')
ax[0].axhline(1, ls=':', c='k')

sns.scatterplot(data=clean8, x='inflow_to_band', y='model_storage_kge',
                hue='path', palette=pal_path, ax=ax[1], legend=False)
ax[1].set(xscale='log', xlabel='inflow-to-band ratio (log)', ylabel='storage KGE', title='sweet spot')
ax[1].axhline(0, ls=':', c='gray'); ax[1].axhline(KGE_REF, ls='--', c='r', lw=.8)

plt.tight_layout(); plt.savefig(file_path + 'res8_inflow_to_band.pdf'); plt.show()

In [ ]:
# ===== FIG res8-PCR — same as res8 but PCR-GLOBWB instead of the point model =====
clean8 = metrics_df[metrics_df['metric_ok']].dropna(subset=['inflow_to_band'])
fig, ax = plt.subplots(1, 2, figsize=(13, 5))

sns.scatterplot(data=clean8, x='inflow_to_band', y='pcr_storage_alpha',
                hue='path', palette=pal_path, ax=ax[0])
ax[0].set(xscale='log', xlabel='inflow-to-band ratio (log)', ylabel='PCR storage α',
          title='variability vs ratio (PCR)')
ax[0].axhline(1, ls=':', c='k')

sns.scatterplot(data=clean8, x='inflow_to_band', y='pcr_storage_kge',
                hue='path', palette=pal_path, ax=ax[1], legend=False)
ax[1].set(xscale='log', xlabel='inflow-to-band ratio (log)', ylabel='PCR storage KGE',
          title='sweet spot (PCR)')
ax[1].axhline(0, ls=':', c='gray'); ax[1].axhline(KGE_REF, ls='--', c='r', lw=.8)

plt.tight_layout(); plt.savefig(file_path + 'res8_inflow_to_band_PCR.pdf'); plt.show()

In [ ]:
# ===== FIG res8 — convergence: point model vs PCR on the same axes =====
clean8 = metrics_df[metrics_df['metric_ok']].dropna(subset=['inflow_to_band'])
clean8 = clean8[clean8['inflow_to_band'] > 0]          # <-- add this line
bins = np.logspace(np.log10(clean8.inflow_to_band.min()), np.log10(clean8.inflow_to_band.max()), 7)

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
for y, a, ttl in [('alpha', ax[0], 'storage α vs ratio'), ('kge', ax[1], 'storage KGE vs ratio')]:
    for src, col, lab in [('model', '#d62728', 'point model'), ('pcr', '#1f77b4', 'PCR-GLOBWB')]:
        a.scatter(clean8['inflow_to_band'], clean8[f'{src}_storage_{y}'], color=col, alpha=.35, s=25, label=lab)
        med = clean8.groupby(pd.cut(clean8.inflow_to_band, bins), observed=True)[f'{src}_storage_{y}'].median()
        a.plot([iv.mid for iv in med.index], med.values, '-o', color=col, lw=2)
    a.set(xscale='log', xlabel='inflow-to-band ratio (log)', title=ttl)
ax[0].axhline(1, ls=':', c='k'); ax[0].set_ylabel('storage α'); ax[0].legend(fontsize=8)
ax[1].axhline(0, ls=':', c='gray'); ax[1].axhline(KGE_REF, ls='--', c='r', lw=.8); ax[1].set_ylabel('storage KGE')
plt.tight_layout(); plt.savefig(file_path + 'res8_model_vs_pcr_convergence.pdf'); plt.show()

In [ ]:
# after the fixed cell 29 adds inflow_to_band:
print(metrics_df['inflow_to_band'].notna().sum(), 'dams with ratio')   # want ~all
metrics_df.to_csv('357metrics.csv', index=False)

In [ ]:
# ===== FIG res8 — inflow-to-band by region: point model vs PCR =====
# tag region if not already present
if 'region' not in metrics_df.columns:
    west_ids = set(pd.read_csv('usable_dams.csv')['geodar'])
    ids = metrics_df['dam_id'] if 'dam_id' in metrics_df.columns else metrics_df.index
    metrics_df['region'] = np.where(pd.Series(ids).isin(west_ids).values, 'west', 'Mississippi')

clean8 = metrics_df[metrics_df['metric_ok'] & (metrics_df['inflow_to_band'] > 0)]
regions = ['west', 'Mississippi']
panels  = [('alpha', 'storage α'), ('kge', 'storage KGE')]

fig, ax = plt.subplots(2, 2, figsize=(13, 9))
for i, reg in enumerate(regions):
    g = clean8[clean8['region'] == reg]
    bins = np.logspace(np.log10(g.inflow_to_band.min()), np.log10(g.inflow_to_band.max()), 6)
    for j, (y, ylab) in enumerate(panels):
        a = ax[i, j]
        for src, col, lab in [('model', '#d62728', 'point model'), ('pcr', '#1f77b4', 'PCR-GLOBWB')]:
            a.scatter(g['inflow_to_band'], g[f'{src}_storage_{y}'], color=col, alpha=.35, s=22, label=lab)
            med = g.groupby(pd.cut(g.inflow_to_band, bins), observed=True)[f'{src}_storage_{y}'].median()
            a.plot([iv.mid for iv in med.index], med.values, '-o', color=col, lw=2)
        a.set(xscale='log', xlabel='inflow-to-band ratio (log)', ylabel=ylab,
              title=f'{reg} — {ylab}')
        if y == 'alpha':
            a.axhline(1, ls=':', c='k'); a.set_ylim(0, 4)            # clip the α outliers so the trend reads
        else:
            a.axhline(0, ls=':', c='gray'); a.axhline(KGE_REF, ls='--', c='r', lw=.8); a.set_ylim(-2.5, 1)
ax[0, 0].legend(fontsize=8)
fig.suptitle('Inflow-to-band ratio governs storage variability in both basins')
plt.tight_layout(); plt.savefig(file_path + 'res8_by_region.pdf'); plt.show()

In [ ]:
clean8 = metrics_df[metrics_df['metric_ok'] & (metrics_df['inflow_to_band'] > 0)]
fig, ax = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for a, reg in zip(ax, ['west', 'Mississippi']):
    g = clean8[clean8['region'] == reg]
    bins = np.logspace(np.log10(g.inflow_to_band.min()), np.log10(g.inflow_to_band.max()), 6)
    for src, col, lab in [('model', '#d62728', 'point model'), ('pcr', '#1f77b4', 'PCR-GLOBWB')]:
        a.scatter(g['inflow_to_band'], g[f'{src}_storage_alpha'], color=col, alpha=.35, s=22, label=lab)
        med = g.groupby(pd.cut(g.inflow_to_band, bins), observed=True)[f'{src}_storage_alpha'].median()
        a.plot([iv.mid for iv in med.index], med.values, '-o', color=col, lw=2)
    a.set(xscale='log', xlabel='inflow-to-band ratio (log)', title=reg, ylim=(0, 4))
    a.axhline(1, ls=':', c='k')
ax[0].set_ylabel('storage α'); ax[0].legend(fontsize=8)
fig.suptitle('Inflow-to-band ratio vs storage variability — both basins')
plt.tight_layout(); plt.savefig(file_path + 'res8_alpha_by_region.pdf'); plt.show()

In [ ]:
# ===== TABLE: inflow-to-band ratio — Spearman correlations & quartile summary =====
from scipy.stats import spearmanr

# region tag (consistent with the res8 cells)
if 'region' not in metrics_df.columns:
    west_ids = set(pd.read_csv('usable_dams.csv')['geodar'])
    ids = metrics_df['dam_id'] if 'dam_id' in metrics_df.columns else metrics_df.index
    metrics_df['region'] = np.where(pd.Series(ids).isin(west_ids).values, 'west', 'Mississippi')

# usable reservoirs with a defined positive ratio (same filter as res8)
clean8 = metrics_df[metrics_df['metric_ok'] & (metrics_df['inflow_to_band'] > 0)].copy()

def _spear(d, x, y):
    d = d[[x, y]].replace([np.inf, -np.inf], np.nan).dropna()
    if len(d) < 3:
        return np.nan, len(d)
    return spearmanr(d[x], d[y]).correlation, len(d)

# ---- 1) Spearman: inflow-to-band ratio vs storage variability (alpha) ----
rows = []
for reg, d in [('pooled', clean8),
               ('west', clean8[clean8.region == 'west']),
               ('Mississippi', clean8[clean8.region == 'Mississippi'])]:
    rm, n = _spear(d, 'inflow_to_band', 'model_storage_alpha')
    rp, _ = _spear(d, 'inflow_to_band', 'pcr_storage_alpha')
    row = {'region': reg, 'n': n,
            'rho_model_storage_alpha': round(rm, 2),
            'rho_pcr_storage_alpha':   round(rp, 2)}
    # also the storage-KGE correlation (monotonic part; note KGE is hump-shaped)
    km, _ = _spear(d, 'inflow_to_band', 'model_storage_kge')
    kp, _ = _spear(d, 'inflow_to_band', 'pcr_storage_kge')
    row['rho_model_storage_kge'] = round(km, 2)
    row['rho_pcr_storage_kge']   = round(kp, 2)
    rows.append(row)
spearman_table = pd.DataFrame(rows).set_index('region')

# ---- 2) Quartile summary: median storage alpha & KGE by ratio quartile ----
def _quartile_summary(d):
    d = d.dropna(subset=['inflow_to_band']).copy()
    d['q'] = pd.qcut(d['inflow_to_band'], 4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
    g = d.groupby('q', observed=True)
    return pd.DataFrame({
        'n':              g.size(),
        'ratio_median':   g['inflow_to_band'].median().round(2),
        'model_alpha_med': g['model_storage_alpha'].median().round(2),
        'model_kge_med':   g['model_storage_kge'].median().round(2),
    })

quartile_pooled      = _quartile_summary(clean8)
quartile_west        = _quartile_summary(clean8[clean8.region == 'west'])
quartile_mississippi = _quartile_summary(clean8[clean8.region == 'Mississippi'])

print('Usable reservoirs with a defined ratio: n =', len(clean8),
      f"(west {int((clean8.region=='west').sum())}, "
      f"Mississippi {int((clean8.region=='Mississippi').sum())})")
print('\n--- Spearman rho: inflow-to-band ratio vs storage metrics ---')
print(spearman_table)
print('\n--- POOLED: storage alpha & KGE by ratio quartile ---')
print(quartile_pooled)
print('\n--- WEST: storage alpha by ratio quartile ---')
print(quartile_west)
print('\n--- MISSISSIPPI: storage alpha by ratio quartile ---')
print(quartile_mississippi)

# save for the write-up
spearman_table.to_csv('res8_spearman_table.csv')
quartile_pooled.to_csv('res8_quartile_pooled.csv')

spearman_table
